```{tip}
**For the presenter:** before starting the slideshow, run *Kernel → Restart Kernel and Run All Cells*, so all widgets are shown.
```

In [ ]:
%matplotlib inline
# Load the prepared Acetone process, reference and estimation GUI.
%run ../resources/optimization_intro.py
%run ../resources/parameter_fitting_setup.py
import pandas as pd
gui_data = pd.read_excel("experimental_data/tracer_1.xlsx", index_col=0)
estimation_widget = porosity_estimation_widget(
    np.column_stack([gui_data.index, gui_data.iloc[:, 0]]),  # time / s, first column: UV signal
    configuration=acetone_estimation_configuration(),
    label="Acetone tracer", porosity_bounds=(0.1, 0.8),
)

# 04.2 Parameter Fitting

## Contents

**Tools:** CADET-GUI widget: parameter estimation; CADET-Process (Python).

**Python used:** Data loading, column selection, parameter targets and simulation results.

**Introduces:** `ReferenceIO`, `Comparator`, `RMSE`, `add_evaluator()`, simulation dependencies with `requires` and staged characterization with fixed and fitted parameters.

1. The tracer fit in the estimation GUI
2. Load a Python reference and choose a difference metric
3. Connect simulations and fit porosity and dispersion
4. Verify the fitted curve
5. From single fits to a characterization

Optimization Essentials introduced variables, objectives, bounds and the optimizer. Here, we estimate **porosity and axial dispersion together**: first with a fit in the GUI, then by building the full-curve fit in Python with a reference, comparison metric and simulation evaluator.

Porosity controls the mean transport delay; dispersion affects peak spreading. Both influence the curve, so their combined fit matters, not the arrival time alone.

| Fixed | Estimated from the tracer curve |
|---|---|
| Column geometry, flow rate and injection | `column.total_porosity` and `column.axial_dispersion` |

Instrument transport must be accounted for when present; the last part of this lesson shows how an accepted instrument result is carried into the column fit.

## The tracer experiment

The supplied [Acetone dataset](./experimental_data/tracer_1.xlsx) represents a **1 s nonbinding tracer injection** through a **1.4 cm × 1 cm column** at **1 mL/min**. The workbook contains **time in seconds** and **tracer concentration in mM**, sampled every **0.5 s for 180 s**. The GUI and Python sections use this same dataset and column model.

## Fit the tracer curve in the GUI

The estimation widget shows the whole fit in one place, as a short demonstration. The measurement is preloaded (**Import CSV** loads other data). The model starts at **porosity 0.50** and **dispersion `1e-6` m²/s**. Under **Configure parameter space**, **Total porosity** (0.1–0.8) and **Axial dispersion** (`1e-9`–`1e-6` m²/s) are selected. **Run estimation** adjusts both to reduce the curve difference, **Preview** compares the initial and fitted curves, and **Accept** applies the values.

The Python part below builds the same fit step by step, starting from an independent model.

In [ ]:
# Show data import, parameter choices and the prepared Acetone fit.
estimation_widget.display()

## Load the reference in Python

The widget compared the complete curves. In Python, `ReferenceIO`, a difference metric and `Comparator` make that connection explicit.

`pandas.read_excel()` loads the workbook, with time as the index; its concentration column is then wrapped in `ReferenceIO`.

In [ ]:
# Load the workbook
import pandas as pd
data = pd.read_excel("experimental_data/tracer_1.xlsx", index_col=0)

## Store the measured curve with `ReferenceIO`

To quantify the difference between simulation and reference, **CADET-Process** provides a [`comparison`](https://cadet-process.readthedocs.io/en/latest/user_guide/process_evaluation/comparison.html) module.

`ReferenceIO` stores measured times and concentrations for a stream. Time is given in **seconds** and concentrations in the **same units as the model**. Here, the workbook's mM values correspond to mol/m³.

In [ ]:
# Reference from the measured curve


`ReferenceIO.plot()` shows the concentration curve:

In [ ]:
# Plot the reference
_ = reference.plot()

## Connect the reference and simulation with `Comparator`

`Comparator` connects a reference and a difference metric to a simulated signal. Its `plot_comparison()` and `evaluate()` methods let us inspect the match and calculate its score.

In [ ]:
# Comparator


```{note}
Each difference metric carries its reference data. One comparator can combine metrics for several experiments or sensors.
```

## Choose a difference metric

The root mean squared error (`RMSE`) measures the typical concentration difference between simulation and reference. **Zero** means agreement at the compared times. We use it for this fit.

The GUI uses the sum of squared errors (`SSE`). RMSE expresses the discrepancy in concentration units; both reward closer curve agreement.

`Shape` provides correlation and time-offset measures for peak comparisons. The metric should answer the experimental question, and known instrument delays belong in the model.

Construct the metric with its reference data, then register the metric instance and the solution path. Here, `"column.outlet"` selects the simulated column outlet to compare with the measurement.

In [ ]:
# RMSE metric at the column outlet


## Where the process model comes from

`acetone_process()` is a workshop helper defined in [parameter_fitting_setup.py](../resources/parameter_fitting_setup.py), which the notebook loads at the start. It builds a new CADET-Process `Process` from the experimental setup, using the components, unit operations, flow sheet and events introduced in the fundamentals lessons:

**Inlet (Acetone pulse) → nonbinding column → Outlet**

| Part of the setup | Configuration in the helper |
|---|---|
| Component | One nonbinding tracer, **Acetone**: small enough to enter the particle pores, so the fit gives the **total** porosity |
| Inlet | **1 mM for 1 s**, then tracer-free liquid, at **1 mL/min** |
| Column | `LumpedRateModelWithoutPores`, **1.4 cm long × 1 cm diameter**, without binding |
| Observation time | **180 s**, matching the reference dataset |
| Initial estimates | Porosity **0.50**, dispersion **`1e-6` m²/s** |

This is the same experiment and initial model used in the GUI. Each call builds an independent process, so the Python fit starts from these estimates even after accepting a GUI fit.

```{note}
`reference` contains the measured tracer curve. `process` describes the experiment and predicts a curve when simulated. Fitting changes the process's porosity and dispersion to bring that prediction closer to the reference.
```

In [ ]:
# Build the process from the shared recipe and simulate its initial estimates.
process = acetone_process()
from CADETProcess.simulator import Cadet
simulator = Cadet()
simulation_results = simulator.simulate(process)

`Comparator.plot_comparison()` overlays the simulation and reference and displays the selected metric:

In [ ]:
# Compare simulation and reference
_ = comparator.plot_comparison(simulation_results)

`Comparator.evaluate()` returns the metric value for this simulation. This is the score we want to reduce:

In [ ]:
# Evaluate the metric
metrics = comparator.evaluate(simulation_results)
print(metrics)

The starting model does not reproduce the measured curve. The fit will adjust both transport parameters while keeping the experimental conditions fixed.

## Fit two transport parameters

`add_evaluation_object(process)` registers the model. Each `add_variable()` call links a fitted parameter to its column attribute and sets its bounds.

In [ ]:
# Reuse the optimization setup, now with two column parameters.
from CADETProcess.optimization import OptimizationProblem
optimization_problem = OptimizationProblem('porosity_axial_dispersion')

optimization_problem.add_evaluation_object(process)

The two `parameter_path` values connect the variables to the column. `normalization="auto"` accounts for their different scales: porosity is a fraction, while dispersion is in m²/s. Each variable is added with `add_variable(name, targets=[process], parameter_path=..., lb=..., ub=..., normalization="auto")`:

| `name` | `parameter_path` | `lb` | `ub` |
|---|---|---:|---:|
| `porosity` | `flow_sheet.column.total_porosity` | 0.1 | 0.8 |
| `axial_dispersion` | `flow_sheet.column.axial_dispersion` | `1e-9` | `1e-6` m²/s |

In [ ]:
# Porosity and dispersion variables


## Simulate before calculating the fitting error

The tracer visualization in Essentials showed the error for prepared candidate curves. Here, each candidate needs a new **simulation result**. `add_evaluator(simulator)` registers that simulation step.

**Candidate parameters → update process → simulate → compare with reference → RMSE**

In [ ]:
# Register the simulator


`requires=[simulator]` passes the simulator’s result to the `Comparator`. The comparator calculates the registered RMSE; `n_objectives=comparator.n_metrics` declares how many scores it returns. Here, there is **one objective** for **two fitted parameters**.

In [ ]:
# Comparator as objective


## Run the fit

This example uses `U_NSGA3`, which searches with a population of candidates. The supplied run uses **32 candidates**, at most **10 generations** and **8 cores**. Reduce `n_cores` for a smaller machine. This demonstration budget does not establish convergence; compare the selected curve with the data afterwards.

Algorithm comparisons are available in [99.4 Optimization Basics](../../99%20Misc/99.4%20Optimization%20Basics/99.4a_optimization_basics_lesson.ipynb).

In [ ]:
from CADETProcess.optimization import U_NSGA3
optimizer = U_NSGA3()
optimizer.n_cores = 8
optimizer.pop_size = 32
optimizer.n_max_iter = 10

In [ ]:
optimization_results = optimizer.optimize(
        optimization_problem,
        use_checkpoint=False
    )

## Verify the fitted curve

The selected values in `.x` update the process, which is then simulated again. The process may otherwise still contain the last candidate evaluated during the search.

In [ ]:
# Apply the selected transport parameters and inspect their joint fit.
process.flow_sheet.column.total_porosity = float(optimization_results.x[0, 0])
process.flow_sheet.column.axial_dispersion = float(optimization_results.x[0, 1])
print("Fitted porosity:", process.flow_sheet.column.total_porosity)
print("Fitted dispersion / m²/s:", process.flow_sheet.column.axial_dispersion)
fitted_results = simulator.simulate(process)
_ = comparator.plot_comparison(fitted_results)

**Arrival time, peak width and tailing** show the quality of the fit. A small overall error can still hide systematic differences. Parameters at their bounds or a poor curve match call for checking the model, data and search budget before accepting the fit.

The convergence plot shows how RMSE changes during the search. A falling score is useful evidence of progress; it does not replace the curve check.

In [ ]:
_ = optimization_results.plot_convergence('objectives')

## From single fits to a characterization

The Acetone fit had only the column between inlet and detector. Fitting data from a laboratory instrument needs two more considerations:

1. **A fitted parameter absorbs everything the model leaves out.** Parts of the setup that the model does not describe end up in the fitted values, so known contributions are modelled and held fixed.
2. **A measurement only determines parameters it is sensitive to.** Each parameter is fitted from an experiment that contains information about it.

Together, they lead to a characterization **in stages**. The synthetic RTD experiments from 02.4 show both: tracer flows at **6 mL/min** through mixer, tubing and a **10 cm × 2 cm column** to the detector; in the bypass experiment, the column is disconnected.

In [ ]:
# Load the RTD measurements and the supplied delay calculations from 02.4.
%run ../resources/characterization_data.py
pulse_data, step_data = load_rtd_measurements()
flow_rate = 1e-7  # 6 mL/min
measured_system_delay = pulse_delay(pulse_data)
measured_total_delay = breakthrough_delay(step_data)
characterization_view = characterization_setup_widget()

In [ ]:
# Show the flow paths of the column and bypass experiments.
display(characterization_view)

### Which part of the delay belongs to the column?

**Measured mean delay = mean residence time in the mixer and tubing + mean residence time in the column**

| Experiment | What the signal tells us |
|---|---|
| 5 s pulse, column bypassed | About **30 s** instrument delay, corresponding to **3 mL** system volume |
| Continuous tracer feed, column connected | About **140 s** total delay: instrument and column together |

If all 140 s are assigned to the column, the fitted porosity also accounts for the liquid in the mixer and tubing: about **0.45** instead of **0.35**. The delay is matched either way; only the separate bypass run shows which value describes the column.

In [ ]:
# Both measurements and the instrument share of the total delay.
_ = plot_rtd_measurements(pulse_data, step_data)
_ = plot_delay_contributions(3e-6 / flow_rate, measured_total_delay)

### Fit in stages

Each stage fits only the parameters its experiment carries information about, and its accepted values stay fixed in the following stages:

| Stage | Experiment | Fitted | Fixed from earlier stages |
|---|---|---|---|
| System transport | Bypass tracer pulse | System volume | Flow rate and geometry |
| Column transport | Nonbinding tracer | Porosity, dispersion | System transport |
| Binding | Breakthroughs and elution gradients | Binding parameters | System and column transport |

Binding parameters are fitted in the same way as porosity and dispersion: a reference and a comparator per experiment, variables pointing to the binding model's parameters, the simulator as evaluator. One experiment is not enough there, so several (breakthrough, gradients of different length) share one parameter set; **99.5 Advanced Optimization: Exercises** shows the complete fit.

The total delay alone cannot separate **system volume** from **column porosity**: increasing either increases the delay. Nor does it determine **dispersion**: two dispersions can give the same delay but different curve shapes, which only a full-curve metric such as RMSE distinguishes.

```{note}
**In the exercises:** [Exercise 3](./04.2b_parameter_fitting_exercises.ipynb) applies both to a newly packed column. It fits the porosity with the accepted system volume held fixed, compares the result with the value obtained without the system run, and checks whether the delay fit can also determine dispersion.
```

## Summary

- `ReferenceIO` stores the measured curve; `Comparator` connects it to a difference metric.
- Variables change porosity and dispersion in the process; the simulator supplies each candidate curve.
- The selected parameters are verified by simulating again and checking the complete curve.
- A characterization fits in stages: accepted values stay fixed, so each parameter describes its own part of the setup.

**Now:** [**04.2 Parameter Fitting: Exercises**](04.2b_parameter_fitting_exercises.ipynb): compare tracer experiments, fit transport parameters and characterize a newly packed column; optionally, compare binding experiments.

**Next lesson:** [**04.3 Fractionation**](../04.3%20Fractionation/04.3a_fractionation_lesson.ipynb) uses the simulated chromatogram to define product pools and evaluate their performance.